# Suite TYP — Types

`Types` renders schemas as Python dataclasses and reads dataclasses back as schemas, step by step: a session runs over
the schemas a store registers, Python's syntax trees and the output, which holds the module. `Dataclass` takes one
decision per object schema, whether its class is frozen; `Schema` reads one class back. Both round trips are tested:
schemas to source to schemas, and source to schemas to source.

In [ ]:
import { Types } from "@mbse/codegen-python";
import { Transforms as T } from "@mbse/patterns";
import { Python312, Syntax as Py } from "@mbse/programs/Python";
import { JSON, Modules, Proxies, Schemas as S, Validators as V } from "@mbse/schemas/Framework";
import { ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, native, raises, same, stocked } from "./support.js";

const B = Py.LANGUAGE.Builders as any;
const Phone = new S.OfObject.Builder().name("Phone").properties(native("number")).create();
const Contact = new S.OfObject.Builder().name("Contact").ref().description('A "person"\nwith a home').properties(
  native("name"), native("age", BigInt), (p) => p.name("home").of(Phone),
  (p) => p.name("tags").of((t) => t.as_indexed((i) => i.of((x) => x.as_native(String)))),
  (p) => p.name("phones").of((t) => t.as_indexed((i) => i.of(Phone)))).create();
const Empty = new S.OfObject.Builder().name("Empty").create();

/** The schemas as a module's JSON text, to compare schemas by what they say. */
function module_of(schemas: unknown[]): string {
  const store = new Proxies.OfStore();
  return JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, schemas as never) as never);
}

/** The schemas a reading session's store registers, by name. */
function registered(session: T.Session, ...names: string[]): any[] {
  const schemas = (session.store as any).stores[0].store;
  return names.map((name) => schemas.registered(name));
}

const SOURCE = `from __future__ import annotations
from dataclasses import dataclass


@dataclass(eq=False)
class Contact:
    "A \\"person\\"\\nwith a home"
    name: str | None = None
    age: int | None = None
    home: Phone | None = None
    tags: list[str] | None = None
    phones: list[Phone] | None = None


@dataclass
class Empty:
    pass


@dataclass(frozen=True)
class Phone:
    number: str | None = None
`;

## TYP-01 · The store: the schemas a store registers and refers to, Python's syntax trees, and the output holding the module

In [ ]:
{
  const module = B.Module().create();
  const store = Types.store(stocked(Contact, Empty), module);
  assert(same(store.extent("Schemas.Object"), [Contact, Empty, Phone])); // Phone: referred to
  const output = store.singleton(Types.OUTPUT) as unknown as Types.Output;
  assert(output.module === module && output.schema_name() === "Codegen.Output" && output.owner() === null);
  assert(output.identity() === output.identity() && output.identity() !== new Types.Output().identity());
  assert(same((V.entries_of(output as never).get("modules") ?? []).map((e) => e.targets.get("module")), [module])
    && V.entries_of(new Types.Output() as never).size === 0);
  assert(Types.Output.Schema.singleton === "Codegen.Output" && same(Types.Generated.links, ["output", "module"]));
  assert(same(store.extent("Programs.Python.Module"), [module])); // reached from the output
}

## TYP-02 · Dataclass: a candidate per value of `frozen` for each object schema it renders and has no class for

In [ ]:
{
  const session = new T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), [...Types.TO_PYTHON]);
  assert(equal(session.candidates().map((c) => [(c.match["s"] as any).name, c.arguments]), [
    ["Contact", { frozen: false }], ["Contact", { frozen: true }], ["Empty", { frozen: false }],
    ["Empty", { frozen: true }], ["Phone", { frozen: false }], ["Phone", { frozen: true }]]));
  assert(Types.Dataclass.parameters.get("frozen")?.description === "Whether the class is frozen");
  const Link = new S.OfRelation.Builder().name("Link").links("a", "b").create();
  const Word = new S.OfNative.Builder().name("Word").type(BigInt).create();
  const Reach = new S.OfUnion.Builder().name("Reach").branches((b) => b.name("phone").of(Phone)).create();
  /** A list of lists, `depth` deep. */
  const nested = (depth: number, item: any): any => depth === 0 ? item : (t: any) => t.as_indexed((i: any) => i.of(nested(depth - 1, item)));
  const holding = (name: string, spec: any, field = "x") =>
    new S.OfObject.Builder().name(name).properties((p) => p.name(field).of(spec)).create();
  const unrendered = [
    holding("Inline", (t: any) => t.as_object((o: any) => o)), holding("Foreign", (t: any) => t.as_native(
      (n: any) => n.token("ccpp", "int32_t"))), holding("Bounded", (t: any) => t.as_indexed((i: any) => i.of(Phone).extent({ minimum: 0n, maximum: 9n }))),
    holding("Named", Word), holding("Union", (t: any) => t.as_union((u: any) => u.branches((b: any) => b.name("phone").of(Phone)))), holding("Deep", nested(Types.DEPTH + 1, Phone)),
    holding("ByPhone", (t: any) => t.as_indexed((i: any) => i.key(Phone).of(Phone))),
    new S.OfObject.Builder().name("Generic").parameters((q) => q.name("n")).create()];
  const others = new T.Session(Types.store(stocked(...unrendered), B.Module().create()), [...Types.TO_PYTHON]);
  assert(same([...new Set(others.candidates().map((c) => (c.match["s"] as any).name))], ["Phone"])); // referred to; none of the others renders
  assert(Types.missing(others).length === unrendered.length + 1 && others.run(Types.PLAIN) === 1 && others.done);
  assert(same(Types.missing(others).map((s: any) => s.name), unrendered.map((s) => s.name as string).sort())); // completeness: what has no class
}

## TYP-03 · Taking the steps: the caller's decisions and the policy's, the module printed, the trace

In [ ]:
{
  const session = new T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), [...Types.TO_PYTHON]);
  const phone = session.candidates().find((c) => c.match["s"] === Phone && c.arguments["frozen"] === true) as T.Candidate;
  session.take(phone); // the caller decides that Phone is frozen
  assert(session.run(Types.PLAIN) === 2 && session.done);
  assert(Types.text(session) === SOURCE);
  assert(equal(session.steps.map((step) => [step.transform, step.match, step.arguments, step.by]), [
    ["Dataclass", [["s", "Phone"]], [["frozen", true]], "caller"],
    ["Dataclass", [["s", "Contact"]], [["frozen", false]], "policy"],
    ["Dataclass", [["s", "Empty"]], [["frozen", false]], "policy"]]));
  assert(Types.text(Types.generate(stocked(Contact, Empty))) === SOURCE.replace("@dataclass(frozen=True)", "@dataclass"));
  const frozen = Types.generate(stocked(Contact), new T.Policy(new T.Clause("Dataclass", { frozen: true })));
  assert(Types.text(frozen).includes("@dataclass(eq=False, frozen=True)\nclass Contact:"));
  assert(Py.LANGUAGE.validate((session.store.singleton(Types.OUTPUT) as unknown as Types.Output).module as never).length === 0); // a valid tree
}

## TYP-04 · Schema: a dataclass read back as a schema; names, references, lists, docstrings, `eq=False`; refusals

In [ ]:
{
  const reading = Types.read(Python312.parse(SOURCE) as never);
  const [contact, empty, phone] = registered(reading, "Contact", "Empty", "Phone");
  assert(contact.ref === true && contact.description === 'A "person"\nwith a home'
    && same([...contact.properties.keys()], ["name", "age", "home", "tags", "phones"]));
  assert(contact.properties.get("home").type === phone && contact.properties.get("phones").type.item === phone); // filled when read
  assert(contact.properties.get("age").type.equals(new S.OfNative.Data(BigInt))
    && contact.properties.get("tags").type.item.equals(new S.OfNative.Data(String)));
  assert(empty.properties.size === 0 && phone.ref === false && same([...phone.properties.keys()], ["number"]));
  assert(same(reading.steps.map((step) => step.transform), ["Schema", "Schema", "Schema"]) && reading.done);
  const written = `@dataclass
class Plain:
    """A 'quoted' \\\\ text"""
    count: int
    note: str = "x"


class Ignored:
    x: int


@dataclass(frozen=True, eq=True)
class Frozen:
    f()
    x: int


@other
class Other:
    x: int
`;
  const target = stocked(Phone);
  Types.read(Python312.parse(written) as never, target);
  assert((target.registered("Plain") as any).description === "A 'quoted' \\ text" && !(target.registered("Frozen") as any).ref);
  assert(same([...(target.registered("Plain") as any).properties.keys()], ["count", "note"]) && !target.names().includes("Ignored"));
  assert((target.registered("Frozen") as any).description === null && !target.names().includes("Other"));
  assert(target.names().includes("Phone")); // a store given is the one read into
  assert(Types.read(Python312.parse("@dataclass\nclass Single:\n    'single'\n") as never).done);
  assert(registered(Types.read(Python312.parse("@dataclass\nclass Single:\n    'it\\'s'\n") as never), "Single")[0].description === "it's");
  for (const annotation of ["dict[str]", "dict[str, int, bool]", "set[int]", "typing.Any", "x.y[int]", "int | str", "int | 3", "int + None"]) {
    raises(ValueError, () => Types.read(Python312.parse(`@dataclass\nclass Bad:\n    x: ${annotation}\n`) as never),
      `Bad.x: cannot read the annotation ${annotation}`);
  }
}

## TYP-05 · Round trips: schemas to source to the same schemas, and source to schemas to the same source

In [ ]:
{
  const back = registered(Types.read(Python312.parse(SOURCE) as never), "Contact", "Empty", "Phone");
  assert(module_of(back) === module_of([Contact, Empty, Phone])); // what the schemas say, frozen aside
  const session = new T.Session(Types.store(stocked(...back), B.Module().create()), [...Types.TO_PYTHON]);
  session.take(session.candidates().find((c) => c.match["s"] === back[2] && c.arguments["frozen"] === true) as T.Candidate);
  session.run(Types.PLAIN);
  assert(Types.text(session) === SOURCE); // the decision the trace keeps, taken again
  assert(Types.text(Types.read(Python312.parse(SOURCE) as never)) === Python312.print(Python312.parse(SOURCE))); // read leaves the module
}

## TYP-06 · Regenerating after the schemas change: decisions taken again by schema name, a new schema asked, a gone one reported

In [ ]:
{
  const first = new T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), [...Types.TO_PYTHON]);
  first.take(first.candidates().find((c) => c.match["s"] === Phone && c.arguments["frozen"] === true) as T.Candidate);
  first.run(Types.PLAIN);
  const Phone2 = new S.OfObject.Builder().name("Phone").properties(native("number"), native("kind")).create(); // a field more
  const Address = new S.OfObject.Builder().name("Address").properties(native("city")).create(); // new
  const Contact2 = new S.OfObject.Builder().name("Contact").ref().properties(
    native("name"), (p) => p.name("home").of(Phone2), (p) => p.name("address").of(Address)).create();
  const again = new T.Session(Types.store(stocked(Contact2), B.Module().create()), [...Types.TO_PYTHON], {}, null, first.steps);
  assert(again.run() === 2 && equal(again.steps.map((step) => [step.key, step.by]), [
    ["Dataclass(s=Contact)", "reused"], ["Dataclass(s=Phone)", "reused"]]));
  assert(same([...new Set(again.candidates().map((c) => (c.match["s"] as any).name))], ["Address"])); // the one decision left to ask
  assert(same(again.orphans.map((step) => step.key), ["Dataclass(s=Empty)"])); // Empty is gone
  again.run(Types.PLAIN);
  assert(Types.text(again).includes("@dataclass(frozen=True)\nclass Phone:\n    number: str | None = None\n    kind: str | None = None\n"));
  const changes = T.diff(first.steps, again.steps);
  assert(same(changes.added.map((s) => s.key), ["Dataclass(s=Address)"]) && same(changes.removed.map((s) => s.key), ["Dataclass(s=Empty)"])
    && changes.changed.length === 0);
  assert(Types.text(Types.generate(stocked(Contact2), Types.PLAIN, first.steps)) === Types.text(again));
}

## TYP-07 · Dicts, nested lists and keyword fields, both ways

In [ ]:
{
  const SHEET = "from __future__ import annotations\nfrom dataclasses import dataclass\n\n\n@dataclass\nclass Score:\n    value: float | None = None\n\n\n@dataclass\nclass Sheet:\n    from_: str | None = None\n    grid: list[list[int]] | None = None\n    scores: dict[str, Score] | None = None\n    tags: dict[int, list[bytes]] | None = None\n";
  const Score = new S.OfObject.Builder().name("Score").properties(native("value", Number)).create();
  const Sheet = new S.OfObject.Builder().name("Sheet").properties(
    native("from"), (p) => p.name("grid").of((t) => t.as_indexed((i) => i.of((x) => x.as_indexed((j) => j.of(
      (y) => y.as_native(BigInt)))))),
    (p) => p.name("scores").of((t) => t.as_indexed((i) => i.key((k) => k.as_native(String)).of(Score))),
    (p) => p.name("tags").of((t) => t.as_indexed((i) => i.key((k) => k.as_native(BigInt)).of(
      (x) => x.as_indexed((j) => j.of((y) => y.as_native(Uint8Array))))))).create();
  const generated = Types.generate(stocked(Sheet));
  assert(Types.text(generated) === SHEET && Types.missing(generated).length === 0);
  const back = registered(Types.read(Python312.parse(SHEET) as never), "Score", "Sheet");
  assert(module_of(back) === module_of([Score, Sheet]) && same([...back[1].properties.keys()], ["from", "grid", "scores", "tags"]));
  assert(Types.text(Types.generate(stocked(...back))) === SHEET);
}

## TYP-08 · Names Python cannot spell: written as the schemas have them, reported by path, and refused when the source is taken

In [ ]:
{
  const holding = (name: string, spec: any, field = "x") =>
    new S.OfObject.Builder().name(name).properties((p) => p.name(field).of(spec)).create();
  const odd = Types.generate(stocked(holding("Bad-Name", Phone), holding("class", Phone), holding("Dashed", Phone, "first-name")));
  assert(odd.done && Types.missing(odd).length === 0); // every schema has its class
  const PROBLEMS = ["body[2].name: 'Bad-Name' is not an identifier", "body[3].body[0].target.id: 'first-name' is not an identifier",
    "body[5].name: 'class' is a keyword"];
  assert(same(Types.problems(odd), PROBLEMS)); // the paths of the names
  raises(ValueError, () => Types.text(odd), "the module is not valid Python: " + PROBLEMS.join("; "));
  assert(Types.problems(Types.generate(stocked(Contact))).length === 0);
}

## TYP-09 · Relations: an entry class per relation, a field per adjacency holding its entries, read alike from proxies and generated classes

In [ ]:
{
  const ENTRIES = "from __future__ import annotations\nfrom dataclasses import dataclass, field\nfrom typing import ClassVar\n\n\n@dataclass(eq=False)\nclass Contact:\n    name: str | None = None\n    phones: tuple[Phones, ...] = ()\n\n\n@dataclass(eq=False)\nclass Pager:\n    owners: tuple[Phones, ...] = ()\n\n\n@dataclass(eq=False)\nclass Parentage:\n    LINKS: ClassVar[tuple[str, ...]] = (\"parent\", \"child\")\n    parent: Person | None = None\n    child: Person | None = None\n\n\n@dataclass(eq=False)\nclass Person:\n    children: tuple[Parentage, ...] = field(default=(), metadata={\"me\": \"parent\"})\n    parents: tuple[Parentage, ...] = field(default=(), metadata={\"me\": \"child\"})\n\n\n@dataclass(eq=False)\nclass Phone:\n    number: str | None = None\n    owners: tuple[Phones, ...] = ()\n\n\n@dataclass(eq=False)\nclass Phones:\n    \"A number to call\"\n    LINKS: ClassVar[tuple[str, ...]] = (\"owner\", \"phone\")\n    UNIQUES: ClassVar[tuple[tuple[str, ...], ...]] = ((\"label\", \"owner\"), (\"phone\",))\n    owner: Contact | None = None\n    phone: Pager | Phone | None = None\n    label: str | None = None\n";
  const Phones = new S.OfRelation.Builder().name("Phones").description("A number to call").links("owner", "phone").properties(
    native("label")).unique("owner", "label").unique("phone").create();
  const Parentage = new S.OfRelation.Builder().name("Parentage").links("parent", "child").create();
  const Card = new S.OfObject.Builder().name("Contact").ref().properties(native("name")).relations(
    (r) => r.name("phones").of(Phones).me("owner")).create();
  const Mobile = new S.OfObject.Builder().name("Phone").ref().properties(native("number")).relations(
    (r) => r.name("owners").of(Phones).me("phone")).create();
  const Pager = new S.OfObject.Builder().name("Pager").ref().relations((r) => r.name("owners").of(Phones).me("phone")).create();
  const Person = new S.OfObject.Builder().name("Person").ref().relations((r) => r.name("children").of(Parentage).me("parent"),
    (r) => r.name("parents").of(Parentage).me("child")).create();
  const MODEL: any[] = [Card, Pager, Person, Mobile, Parentage, Phones];
  const related = Types.generate(stocked(...MODEL));
  assert(Types.text(related) === ENTRIES && Types.missing(related).length === 0);
  assert(same(related.steps.map((step) => step.key).slice(-2), ["Entry(r=Parentage)", "Entry(r=Phones)"])); // a step per entry class
  const back = registered(Types.read(Python312.parse(ENTRIES) as never), ...MODEL.map((schema) => schema.name as string));
  assert(module_of(back) === module_of(MODEL)); // links, properties, uniques, descriptions and adjacencies, as they were
  assert(Types.text(Types.generate(stocked(...back))) === ENTRIES);
  /** Code that reads an adjacency's entries: the same for proxies and generated classes. */
  const numbers = (contact: any) => contact.phones.map((entry: any) => [entry.label, entry.phone.number]);
  const proxies = stocked(...MODEL) as any;
  const ann = proxies.Contact().name("ann").phones((e: any) => e.phone(proxies.Phone().number("1").create()).label("home")).create();
  const twin = { name: "ann", phones: [] as any[] }; // what generated Python classes hold, written as data here
  const phone = { number: "1", owners: [] as any[] };
  const entry = { owner: twin, phone, label: "home" };
  twin.phones = [entry];
  phone.owners = [entry];
  assert(equal(numbers(ann), numbers(twin)) && equal(numbers(ann), [["home", "1"]]) && ann.phones[0].owner === ann);
  // What renders no entry class, and what reads back no relation.
  const Lone = new S.OfRelation.Builder().name("Lone").links("a", "b").create(); // nobody declares b
  const Generic = new S.OfRelation.Builder().name("Generic").links("a", "b").parameters((q) => q.name("n")).create();
  const Reaching = new S.OfRelation.Builder().name("Reaching").links("a", "b").properties((q) => q.name("x").of(
    new S.OfUnion.Builder().branches((b) => b.name("n").of((t) => t.as_native(BigInt))).create())).create();
  const ends = (relation: any) => relation.links.map((link: string) => new S.OfObject.Builder().name(`${relation.name}${link.toUpperCase()}`)
    .ref().relations((q) => q.name("xs").of(relation).me(link)).create());
  assert(same(Types.missing(Types.generate(stocked(...ends(Lone).slice(0, 1)))).map((s: any) => s.name), ["Lone"]));
  assert(same(Types.missing(Types.generate(stocked(...ends(Generic), ...ends(Reaching)))).map((s: any) => s.name), ["Generic", "Reaching"]));
  const Unnamed = new S.OfRelation.Builder().links("a", "b").create();
  assert(same(Types.missing(Types.generate(stocked(new S.OfObject.Builder().name("Holder").relations(
    (q) => q.name("xs").of(Unnamed).me("a")).create()))).map((s: any) => s.name), ["Holder"])); // an unnamed relation has no entry class
  raises(ValueError, () => Types.read(Python312.parse("@dataclass\nclass W:\n    xs: tuple[Unknown, ...] = ()\n") as never),
    "W.xs: Unknown is not a relation the module or the store holds");
  raises(ValueError, () => Types.read(Python312.parse(ENTRIES.replace('field(default=(), metadata={"me": "parent"})', "()")) as never),
    "Person.children: cannot tell which link of Parentage it is from");
  const held = stocked(Phones, Card, Mobile); // a relation the store holds, with no class in the module, so its link is given
  Types.read(Python312.parse('@dataclass\nclass Pager:\n    owners: tuple[Phones, ...] = field(default=(), metadata={"me": "phone"})\n') as never,
    held);
  assert((held.registered("Pager") as any).adjacencies.get("owners").relation === Phones
    && (held.registered("Pager") as any).adjacencies.get("owners").me === "phone");
  raises(ValueError, () => Types.read(Python312.parse("@dataclass\nclass Pager:\n    owners: tuple[Phones, ...] = ()\n") as never,
    stocked(Phones, Card, Mobile)), "Pager.owners: cannot tell which link of Phones it is from");
  const odd = Types.read(Python312.parse( // a link of another annotation types nothing; the link typed by W is the one
    '@dataclass(eq=False)\nclass R:\n    LINKS: ClassVar[tuple[str, ...]] = ("a", "b")\n    a: list[W] | None = None\n'
    + '    b: W | None = None\n\n\n@dataclass\nclass W:\n    xs: tuple[R, ...] = ()\n') as never);
  assert(registered(odd, "W")[0].adjacencies.get("xs").me === "b");
}

## TYP-10 · Unions and intersections: a class of a field per branch or part, read alike from proxies and generated classes

In [ ]:
{
  const VARIANTS = "from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import ClassVar\n\n\n@dataclass(eq=False)\nclass Card:\n    reach: Reach | None = None\n    stamps: list[Stamp] | None = None\n\n\n@dataclass\nclass Phone:\n    number: str | None = None\n\n\n@dataclass\nclass Reach:\n    \"How to reach\"\n    KIND: ClassVar[str] = \"union\"\n    phone: Phone | None = None\n    email: str | None = None\n    from_: str | None = None\n\n\n@dataclass\nclass Stamp:\n    KIND: ClassVar[str] = \"intersection\"\n    when: int | None = None\n    who: Reach | None = None\n";
  const Reach = new S.OfUnion.Builder().name("Reach").description("How to reach").branches(
    (b) => b.name("phone").of(Phone), (b) => b.name("email").of((x) => x.as_native(String)),
    (b) => b.name("from").of((x) => x.as_native(String))).create();
  const Stamp = new S.OfIntersection.Builder().name("Stamp").parts((b) => b.name("when").of((x) => x.as_native(BigInt)),
    (b) => b.name("who").of(Reach)).create();
  const Badge = new S.OfObject.Builder().name("Card").ref().properties((p) => p.name("reach").of(Reach), (p) => p.name("stamps").of(
    (x) => x.as_indexed((i) => i.of(Stamp)))).create();
  const varied = Types.generate(stocked(Badge));
  assert(Types.text(varied) === VARIANTS && Types.missing(varied).length === 0);
  assert(same(varied.steps.map((step) => step.key), ["Dataclass(s=Card)", "Dataclass(s=Phone)", "Union(s=Reach)",
    "Intersection(s=Stamp)"]));
  const back = registered(Types.read(Python312.parse(VARIANTS) as never), "Card", "Phone", "Reach", "Stamp");
  assert(module_of(back) === module_of([Badge, Phone, Reach, Stamp])); // branches, parts, a keyword's, a description
  assert(Types.text(Types.generate(stocked(...back))) === VARIANTS);
  /** Code that reads a union value's branches: the same for proxies and generated classes. */
  const reached = (card: any) => card.reach.phone === null ? card.reach.email : card.reach.phone.number;
  const proxies = stocked(Badge) as any;
  const twin = { reach: { phone: null, email: "a@x" } }; // what a generated Python Reach holds, written as data here
  assert(reached(proxies.Card().reach((u: any) => u.email("a@x")).create()) === reached(twin) && reached(twin) === "a@x");
  const frozen = Types.generate(stocked(Badge), new T.Policy(new T.Clause("Dataclass", { frozen: false }),
    new T.Clause("Union", { frozen: true }), new T.Clause("Intersection", { frozen: true })));
  assert(Types.text(frozen).includes("@dataclass(frozen=True)\nclass Reach:") && Types.text(frozen).includes("@dataclass(frozen=True)\nclass Stamp:"));
  // What renders no class: no branches, parameters, a branch of an inline type.
  const Empty = new S.OfUnion.Builder().name("Empty").create();
  const Generic = new S.OfIntersection.Builder().name("Generic").parameters((q) => q.name("n")).parts(
    (b) => b.name("a").of((x) => x.as_native(BigInt))).create();
  const Inline = new S.OfUnion.Builder().name("Inline").branches((b) => b.name("a").of((x) => x.as_object((o) => o))).create();
  const holder = new S.OfObject.Builder().name("Holder").properties(...[Empty, Generic, Inline].map(
    (t) => (p: any) => p.name((t.name as string).toLowerCase()).of(t))).create();
  assert(same(Types.missing(Types.generate(stocked(holder))).map((s: any) => s.name), ["Empty", "Inline", "Generic"]));
  raises(ValueError, () => Types.read(Python312.parse("@dataclass\nclass W:\n    x: Elsewhere | None = None\n") as never),
    "W.x: Elsewhere is not a class of the module or a schema of the store"); // reading never makes one up
}